In [1]:
from forecasting.model_selection import cross_val
from forecasting.more_models import *
from forecasting.graphs import resid_diagnostic, future_forecast,cross_val_graph
from forecasting.normal_naive_models import naive_pi, drift_pi, mean_pi
import pandas as pd
import plotly.graph_objects as go
from sklearn.metrics import *




class Forecast:

    def __init__(self, data:pd.DataFrame,target_col:str=None, period:int=1):
        
        """
        :param data: pandas.DataFrame - Historical time series data with a date-time index.
        :param target_col: str - Column with historical data. If none will default to the first column.
        :param period: int - Seasonal period.
        """

        self.data = data
        self.period = period

        if not target_col:
            target_col = self.data.columns[0]

        self.target_col = target_col

    def select_best_model(self, n_splits:int=5, test_size:int=None, models:dict=None, time_taken:bool=False):

        """
        Cross validates the models specified.

        
        The model that minimises the mean squared error will be set as the attribute best_model.

        Outputs a data frame that shows the values of the mean absolute error, mean absolute percentage error,
        mean squared error and the max error for the forecast of each model and the observed data.

        Inputs:

            :param n_splits: int - Number of folds.
            :param test_size: int - Forecast horizon during each fold.
            :param models: dict - A dictionary with the models (str) as keys
                                  and their respective forecast functions as values.
            :param time_taken: bool - Toggle woether to print the time taken for each fold of each model

        Outputs:
            pandas.DataFrame: A summary data frame of the metric values for each forecast model.

        """

        #defining a model dictionary to default to
        model_dict = {'naive':naive_pi, 'drift':drift_pi, 'mean':mean_pi,
                      'prophet':prophet_forecast}

        if not models:
            models = model_dict

        #performing cross validation on self.data
        cross_val_frame = cross_val(df=self.data,
                                    target_col=self.target_col,
                                    period=self.period,
                                    n_splits=n_splits,
                                    test_size=test_size,
                                    models=models,
                                    time_taken=time_taken)
        
        #setting the index of the data the same as the cross validation
        first_model = cross_val_frame.index[0][0]
        cross_val_index = cross_val_frame.loc[first_model].index

        obs_data = self.data[self.target_col][cross_val_index]

        #grouping the cross validation data and calculating the metrics for each of the model forecasts
        grouped_frame = cross_val_frame.groupby(by='model')

        output_frame = grouped_frame.agg(mean_absolute_error = ('forecast', lambda x: mean_absolute_error(obs_data,x)),
                                         mean_absolute_percentage_error = ('forecast', lambda x: mean_absolute_percentage_error(obs_data,x)),
                                         mean_squared_error = ('forecast', lambda x: mean_squared_error(obs_data,x)),
                                         max_error = ('forecast', lambda x: max_error(obs_data,x))
                                        )
        
        #setting the model that minimises the mean squared error to self.best_model
        self.best_model = output_frame['mean_squared_error'].idxmin()
        
        return output_frame
    
    
    def plot_diagnostics(self, model:str=None) -> go.Figure:

        """
        A summary of the residual diagnostics.

        This will include a plot of the residuals and their mean value,
        the Autocorrelation funciton (ACF) and its 95% bounds and a histogram
        of the residuals with a theoretical normal distribution.

        The ACF is a collection of the autocorrelation coefficients
        between the residuals and the lagged or shifted residuals.
        A 'good' forecast should produce ACF that is similar to white noise
        or is random, to ensure all information is captured by the forecast.

        Inputs
            :param model: str - Model used to perform the residual diagnostics, a key from model_dict
                                If left None this will default to best_model.
        Outputs:
            go.Figure - Subplots of the residuals, the ACF and a histogram of the residuals.
        """

        if not model:
            model = self.best_model

        return resid_diagnostic(df = self.data,
                                target_col=self.target_col,
                                model = model,
                                period = self.period)
    
    def forecast(self, horizon:int, model:str=None) -> pd.DataFrame:

        """ 
        Creates a summary data frame of the forecast using the model specified.
        The continued dates from the data are the index, and the forecast and 95% and 80%
        prediction intervals are columns.

        Inputs:
            :param horizon: The number of timesteps forecasted into the future.
            :param model: str - Model used to perform the residual diagnostics, a key from model_dict
                                If left None this will default to best_model.
        Ouputs:
            pandas.DataFrame - A forecast using the model specified.

        """

        if not model:
            model = self.best_model

        self.horizon=horizon

        return benchmark_forecast(df = self.data,
                                  target_col=self.target_col,
                                  horizon = horizon,
                                  period = self.period,
                                  model = model,
                                  pred_width=[95,80])
    
    def plot_forecast(self, horizon:int, model:str=None) -> go.Figure:

        """ 
        Creates a plot of the data, the forecast of the desired model
        and the 95% and 80% prediction intervals

        Inputs:
            :param horizon: The number of timesteps forecasted into the future.
            :param model: str - Model used to perform the residual diagnostics, a key from model_dict
                                If left None this will default to best_model.
        Ouputs:
            go.Figure: A plot of the observed data, the forecast and 95% adn 80% prediction intervals.

        """

        if not model:
            model = self.best_model

        return future_forecast(df = self.data,
                               target_col=self.target_col,
                               period = self.period,
                               model = model,
                               horizon=horizon)
    
    
    def plot_cross_validation(self, n_splits:int = 5, test_size:int = None, models:str=None) -> go.Figure:

        """ 
        Cross validates the models specified and plots the results.

        Inputs:
            :param n_splits: int - Number of folds.
            :param test_size: int - Forecast horizon during each fold.
            :param models: dict - A dictionary with the models (str) as keys
                                  and their respective forecast functions as values.
        Outputs:
            go.Figure - A plot of the cross validation forecast and observed data, indicating where each fold is located.
        """

        model_dict = {'naive':naive_pi, 'drift':drift_pi, 'mean':mean_pi,
                      'prophet':prophet_forecast}

        if not models:
            model = model_dict

        return cross_val_graph(df = self.data,
                               target_col=self.target_col,
                               period = self.period,
                               n_splits=n_splits,
                               test_size=test_size,
                               models=models)



C:\Users\danie\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv('https://raw.githubusercontent.com/facebook/prophet/main/examples/example_wp_log_peyton_manning.csv',index_col='ds')

forecast = Forecast(df,period = 7)


In [9]:
models = {'naive':naive_pi(df,'y',period=365,horizon=None), 'drift':drift_pi, 'mean':mean_pi,
                      'prophet':prophet_forecast,'ETS':ETS_forecast, 'ARIMA':ARIMA_forecast}

frame = forecast.select_best_model(n_splits=7,time_taken=True, models=models)
frame

TypeError: unsupported operand type(s) for +: 'NoneType' and 'int'

In [4]:
fig = forecast.plot_diagnostics('naive')
fig.show()

C:\Users\danie\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\numpy\lib\function_base.py:2889: RuntimeWarning:

Degrees of freedom <= 0 for slice

C:\Users\danie\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\numpy\lib\function_base.py:2748: RuntimeWarning:

divide by zero encountered in divide

C:\Users\danie\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\numpy\lib\function_base.py:2748: RuntimeWarning:

invalid value encountered in multiply



The 2-sided chi-squared probability for a normal hypotheis test on the residuals: nan


In [5]:
fig = forecast.plot_cross_validation(models = {'prophet':prophet_forecast,'mean':mean_pi, 'naive':naive_pi})
fig.show()

15:18:56 - cmdstanpy - INFO - Chain [1] start processing
15:18:56 - cmdstanpy - INFO - Chain [1] done processing
15:18:57 - cmdstanpy - INFO - Chain [1] start processing
15:18:57 - cmdstanpy - INFO - Chain [1] done processing
15:18:58 - cmdstanpy - INFO - Chain [1] start processing
15:18:58 - cmdstanpy - INFO - Chain [1] done processing
15:18:59 - cmdstanpy - INFO - Chain [1] start processing
15:18:59 - cmdstanpy - INFO - Chain [1] done processing
15:19:00 - cmdstanpy - INFO - Chain [1] start processing
15:19:00 - cmdstanpy - INFO - Chain [1] done processing


In [ ]:
cross_val(df,'y',7,n_splits = 7,models = {'prophet':prophet_forecast,'mean':mean_pi}).groupby(by=['model','fold']).mean()

14:18:55 - cmdstanpy - INFO - Chain [1] start processing
14:18:55 - cmdstanpy - INFO - Chain [1] done processing
14:18:56 - cmdstanpy - INFO - Chain [1] start processing
14:18:56 - cmdstanpy - INFO - Chain [1] done processing
14:18:56 - cmdstanpy - INFO - Chain [1] start processing
14:18:57 - cmdstanpy - INFO - Chain [1] done processing
14:18:57 - cmdstanpy - INFO - Chain [1] start processing
14:18:58 - cmdstanpy - INFO - Chain [1] done processing
14:18:58 - cmdstanpy - INFO - Chain [1] start processing
14:18:59 - cmdstanpy - INFO - Chain [1] done processing
14:19:00 - cmdstanpy - INFO - Chain [1] start processing
14:19:00 - cmdstanpy - INFO - Chain [1] done processing
14:19:02 - cmdstanpy - INFO - Chain [1] start processing
14:19:02 - cmdstanpy - INFO - Chain [1] done processing


y  forecast     error
model   fold                              
mean    0     7.913326  7.820066  0.093260
        1     8.267760  7.866632  0.401128
        2     8.283577  8.000218  0.283359
        3     8.716504  8.071009  0.645494
        4     8.265313  8.200037  0.065276
        5     8.134316  8.210912 -0.076596
        6     7.711681  8.199974 -0.488293
prophet 0     7.913326  8.812464 -0.899138
        1     8.267760  8.844790 -0.577030
        2     8.283577  8.910957 -0.627380
        3     8.716504  8.584894  0.131610
        4     8.265313  8.356198 -0.090885
        5     8.134316  8.339636 -0.205321
        6     7.711681  7.973490 -0.261810

In [ ]:
fig = forecast.plot_forecast(730,'prophet')
fig.show()

14:19:59 - cmdstanpy - INFO - Chain [1] start processing
14:20:00 - cmdstanpy - INFO - Chain [1] done processing
14:20:02 - cmdstanpy - INFO - Chain [1] start processing
14:20:03 - cmdstanpy - INFO - Chain [1] done processing
14:20:04 - cmdstanpy - INFO - Chain [1] start processing
14:20:05 - cmdstanpy - INFO - Chain [1] done processing


In [ ]:

df = pd.read_csv('../example_validation_data.csv', index_col='ds')
df = df[['y','forecast','fold']]
df['abs error'] = abs(df['y'] - df['forecast'])
df['error'] = df['y'] - df['forecast']


model_dict = {'naive':naive_pi, 'drift':drift_pi, 'mean':mean_pi,
                      'prophet':prophet_forecast}

eval_frame = cross_val(df,'y',7,models = model_dict)
print(eval_frame.index[0][0])

obs_data = df['y'][eval_frame.loc['naive'].index]

grouped_frame = eval_frame.groupby(by='model').agg(mean_absolute_error = ('forecast', lambda x: mean_absolute_error(obs_data,x)),
                                                           mean_absolute_percentage_error = ('forecast', lambda x: mean_absolute_percentage_error(obs_data,x)),
                                                           mean_squared_error = ('forecast', lambda x: mean_squared_error(obs_data,x)),
                                                           max_error = ('forecast', lambda x: max_error(obs_data,x))
                                                           )


grouped_frame


naive:
fold 0: 0.007901430130004883
fold 1: 0.012605667114257812
fold 2: 0.011940717697143555
fold 3: 0.0032961368560791016
fold 4: 0.012516260147094727
drift:
fold 0: 0.03417563438415527
fold 1: 0.05115151405334473
fold 2: 0.08895516395568848
fold 3: 0.1125802993774414
fold 4: 0.15345525741577148
mean:
fold 0: 0.05223250389099121
fold 1: 0.09385848045349121
fold 2: 0.15877270698547363
fold 3: 0.15779781341552734


10:51:11 - cmdstanpy - INFO - Chain [1] start processing


fold 4: 0.2455732822418213
prophet:


10:51:11 - cmdstanpy - INFO - Chain [1] done processing
10:51:11 - cmdstanpy - INFO - Chain [1] start processing


fold 0: 0.5903220176696777


...

10:51:13 - cmdstanpy - INFO - Chain [1] start processing
10:51:13 - cmdstanpy - INFO - Chain [1] done processing


fold 3: 1.0727581977844238


10:51:14 - cmdstanpy - INFO - Chain [1] start processing
10:51:14 - cmdstanpy - INFO - Chain [1] done processing


fold 4: 1.3306159973144531
naive


,mean_absolute_error,mean_absolute_percentage_error,mean_squared_error,max_error
model,,,,
drift,0.963773,0.110969,1.558146,5.844128
mean,0.725698,0.087568,0.868435,4.543638
naive,0.820449,0.096550,1.269413,5.428566
prophet,0.779346,0.094687,1.381744,4.516145


In [ ]:
df = pd.read_csv('../example_validation_data.csv', index_col='ds')
df = df[['y','forecast','fold']]
df['abs error'] = abs(df['y'] - df['forecast'])
df['error'] = df['y'] - df['forecast']

from forecasting.model_selection import cross_val

cross_val(df,'y',7)

naive:
fold 0: 0.13431334495544434
fold 1: 0.013016939163208008
fold 2: 0.008224010467529297
fold 3: 0.01137995719909668
fold 4: 0.0125274658203125
drift:
fold 0: 0.028862714767456055
fold 1: 0.08210229873657227
fold 2: 0.07123565673828125
fold 3: 0.08412051200866699
fold 4: 0.14631938934326172
mean:
fold 0: 0.11517715454101562
fold 1: 0.10312628746032715
fold 2: 0.10162997245788574
fold 3: 0.14696979522705078
fold 4: 0.21506524085998535
ETS:


Exception ignored on calling ctypes callback function: <function ExecutionEngine._raw_object_cache_notify at 0x000001E32B5C7C40>
Traceback (most recent call last):
  File "C:\Users\danie\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\llvmlite\binding\executionengine.py", line 178, in _raw_object_cache_notify
    def _raw_object_cache_notify(self, data):

KeyboardInterrupt: 


fold 0: 42.10662126541138
fold 1: 1.2004261016845703
fold 2: 1.7017631530761719
